<a href="https://colab.research.google.com/github/SanikaPatil1008/Deep_Learning/blob/main/Exp_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# 1. Install Dependencies
!pip install -q torch torchvision onnx onnxruntime onnxscript

# 2. Import Libraries
import torch
import numpy as np
import onnxruntime as ort
from torchvision.models import resnet18, ResNet18_Weights

# 3. Load Pretrained Model
model = resnet18(weights=ResNet18_Weights.DEFAULT)
model.eval()
print(" PyTorch Model Loaded")

# 4. Convert PyTorch → ONNX
dummy_input = torch.randn(1, 3, 224, 224)

torch.onnx.export(
    model,
    dummy_input,
    "resnet18.onnx",
    input_names=['input'],
    output_names=['output'],
    opset_version=18
)

print(" Model Converted to ONNX")

# 5. Load ONNX Model
session = ort.InferenceSession("resnet18.onnx")

# 6. Run ONNX Inference
input_data = dummy_input.numpy()
onnx_output = session.run(None, {'input': input_data})

print(" ONNX Output Shape:", onnx_output[0].shape)

# 7. Compare PyTorch vs ONNX
with torch.no_grad():
    torch_output = model(dummy_input).numpy()

difference = np.mean(np.abs(torch_output - onnx_output[0]))
print(" Difference:", difference)

# 8. Simulated API Prediction
def predict():
    test_input = np.random.randn(1, 3, 224, 224).astype(np.float32)
    output = session.run(None, {'input': test_input})
    return output[0].shape

print(" API Test Output:", predict())

print("\n FULL PIPELINE EXECUTED SUCCESSFULLY ")

 PyTorch Model Loaded


W0503 18:57:36.734000 7959 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0503 18:57:36.735000 7959 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0503 18:57:36.738000 7959 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0503 18:57:36.739000 7959 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
 Model Converted to ONNX
 ONNX Output Shape: (1, 1000)
 Difference: 1.2630743e-06
 API Test Output: (1, 1000)

 FULL PIPELINE EXECUTED SUCCESSFULLY 
